In [0]:
from pyspark.sql.functions import trim, initcap, col, to_date

df = spark.sql(f"SELECT * FROM dbacademy.customers_sales_silver")

df = df.toDF(*[col.strip() for col in df.columns])

df = (df.withColumn("customer_name", initcap(trim(col("customer_name"))))
      .withColumn("customer_id", col("customer_id").cast("long"))
      .withColumn("units_purchased",col("units_purchased").cast("double"))
      .withColumn("total_price",col("total_price").cast("double"))
      .withColumn("order_date",to_date(col("order_date"))))

df = df.dropna(subset=["customer_id", "units_purchased","total_price", "order_date"])


In [0]:
from pyspark.sql.functions import year, month, date_format, round
df = (df.withColumn("order_year",year(col("order_date")))
        .withColumn("order_month",date_format(col("order_date"),'MM')))

df = df.withColumn("avg_price_per_unit", round(col("total_price")/col("units_purchased"),2))

In [0]:
df.write.mode("overwrite").saveAsTable("dbacademy.customers_sales_gold")